<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 6 章：练习解答

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 供 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## 练习 6.1：加入格式感知的奖励塑形

- 如果没有找到 `\boxed{}` 答案，可以使用第 3 章实现的 `fallback="number_then_full"` 回退策略，按如下方式给予部分奖励（0.5 分）：

In [ ]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def partial_reward_rlvr(answer_text, ground_truth):
    
    # 1）尝试提取 boxed 答案
    boxed = extract_final_candidate(
        answer_text, fallback=None
    )
    if boxed:
        correct = grade_answer(boxed, ground_truth)
        return 1.0 if correct else 0.0

    # 2）如果没有找到 boxed 答案，则查找数字
    unboxed = extract_final_candidate(
        answer_text, fallback="number_then_full"
    )
    if unboxed:
        correct = grade_answer(unboxed, ground_truth)
        return 0.5 if correct else 0.0

    return 0.0

- 将其接入第 6 章代码并使用相同设置训练后，部分奖励变体的准确率（37.8%）低于标准 GRPO 设置（47.4%），尽管它的平均回复长度更短。

| # | 方法                                   | 步数 | 最大词元数 | Rollout 数 | 准确率 | 平均词元数 |
|---|------------------------------------------|------|------------|--------------|----------|----------------|
| 1 | GRPO（第 6 章）                         | 50   | 512        | 8            | 47.4%    | 586.11         |
| 2 | GRPO 部分奖励（练习 6.1）      | 50   | 512        | 8            | 37.8%    | 550.33         |

&nbsp;
## 练习 6.2：零优势值情形

- 如果所有奖励都相同（例如全为 0 或全为 1），所有优势值也都会是 0，因为减去均值会消除共同的奖励值，只留下零。

In [ ]:
import torch

rollout_rewards = [0., 0., 0., 0.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

In [ ]:
rollout_rewards = [1., 1., 1., 1.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

- 当所有优势值都为 0 时，损失也为零，因为损失会将优势值与对数概率相乘，而乘以零会消除该项的贡献：

```python
pg_loss = -(advantages.detach() * logps).mean()
```

- 因此，策略梯度为零，模型参数不会针对该提示词更新。

- 这种行为是有意设计的：如果所有 rollout 都同样差或同样好，就没有相对信号告诉模型应强化或抑制哪种行为
- 直观地说，如果模型正确回答了所有问题，就没有必要更新它
- 反过来，如果模型错误回答了所有问题，也不应通过更新模型来强化这种行为
